# 01 - Setup

Install dependencies, download GSM8K, verify GPU, and smoke-test all three models.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q vllm transformers datasets tqdm matplotlib seaborn

## Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. This experiment requires a GPU.")

## Download GSM8K

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_NAME, "main", split=DATASET_SPLIT)
print(f"GSM8K test split: {len(ds)} examples")
print(f"Sample question: {ds[0]['question'][:200]}...")
print(f"Sample answer: {ds[0]['answer'][:200]}...")

## Smoke-test: PRIMARY_MODEL (Qwen3-4B)

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)
sampling = SamplingParams(temperature=0.0, max_tokens=256)

outputs = llm.generate(["What is 2 + 3? Think step by step."], sampling)
print(f"Response: {outputs[0].outputs[0].text[:500]}")
print(f"PRIMARY_MODEL OK")

del llm
import gc; gc.collect()
torch.cuda.empty_cache()

## Smoke-test: CROSS_MODEL (Gemma-3-4B)

In [ ]:
print(f"Loading {CROSS_MODEL}...")
llm = LLM(model=CROSS_MODEL, dtype="bfloat16", max_model_len=4096)
sampling = SamplingParams(temperature=0.0, max_tokens=256)

outputs = llm.generate(["What is 2 + 3? Think step by step."], sampling)
print(f"Response: {outputs[0].outputs[0].text[:500]}")
print(f"CROSS_MODEL OK")

del llm
import gc; gc.collect()
torch.cuda.empty_cache()

## Smoke-test: PARAPHRASER_MODEL (Qwen3-8B)

In [ ]:
print(f"Loading {PARAPHRASER_MODEL}...")
llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
sampling = SamplingParams(temperature=0.0, max_tokens=256)

outputs = llm.generate(["Rewrite this: The cat sat on the mat."], sampling)
print(f"Response: {outputs[0].outputs[0].text[:500]}")
print(f"PARAPHRASER_MODEL OK")

del llm
import gc; gc.collect()
torch.cuda.empty_cache()

print("\nAll models loaded and tested successfully.")